# RAG - Simple Version

A complete RAG system in about 60 lines of code.

This is the same thing as the live class notebook, but simpler. Every cell runs from top to bottom.

---

## The architecture

Real RAG systems have two separate halves. Keep them separate in your head - they run at different times and they fail for different reasons.

```
   INGEST  (runs once, when documents change)

   documents  ->  chunk  ->  embed  ->  store in ChromaDB


   QUERY  (runs every time someone asks something)

   question   ->  embed  ->  search ChromaDB  ->  put chunks in prompt  ->  GPT answers
```

That is the whole system. Everything below is those six boxes.

---

## What changed from the live class version

**One dictionary per chunk instead of four separate lists.**

The live class kept `chunk_texts`, `chunk_titles`, `ids` and `metadatas` as four separate lists that had to stay the same length. When one of them was left empty, ChromaDB crashed with `Non-empty lists are required for ['metadatas']`.

Here each chunk is a single dictionary:

```python
{"id": "...", "title": "...", "text": "..."}
```

Parallel lists cannot get out of sync if there is only one list. This is how you would write it at work.

**One `search()` function that can also filter**, instead of two near-identical functions.

**A `chunk_document()` that returns a plain list**, so there is no tuple unpacking to get wrong.

## Setup

In [ ]:
!pip install -q openai chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 1.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently tak

In [ ]:
import openai
from getpass import getpass

# getpass shows a box. Paste your key there.
# Nothing is saved in the file, so the notebook is safe to share.
openai.api_key = getpass("Paste your OpenAI API key: ")

CHAT_MODEL = "gpt-4o-mini"                 # writes the answers
EMBEDDING_MODEL = "text-embedding-3-small" # turns text into numbers

print("Ready")

## Step 1 - Why we need RAG

Ask the model about our own internal document. It has never seen it.

In [ ]:
question = "In the HireAI MVP plan, which database is used and why?"

response = openai.chat.completions.create(
    model=CHAT_MODEL,
    messages=[{"role": "user", "content": question}]
)

print(response.choices[0].message.content)

The model either admits it does not know, or it invents a confident answer. Inventing is called **hallucination** - the model produces a *plausible* answer, not a *true* one, and it cannot tell you which one you just got.

RAG fixes this by finding the right part of our document and putting it into the prompt first.

## Step 2 - The documents

Each document is a **dictionary** with a title and its text.

Keeping the title next to the text - instead of in a separate `titles` list - is the single change that removes most of the bugs from this notebook.

In [ ]:
documents = [
    {
        "title": "HireAI Architecture Plan",
        "text": """# HireAI — Scalable MVP Architecture Plan (Free-Tier First)

**Goal:** Ship an MVP for the first 100–1,000 users using free AI tools and free-tier infrastructure, built with Claude Code — architected so that migrating to the full paid/scaled stack (the one in `HireAI-Architecture.md`) is a config swap, not a rewrite.

---

## 1. The Core Principle: Modular Monolith + Adapter Boundaries

Do **not** build microservices, Kubernetes, or Kong for 1,000 users. Build **one deployable app** whose internal folders are the exact service boundaries you'll split along later. Every external dependency (LLM, storage, queue, telephony, email) sits behind a small interface ("adapter"), so scaling = swapping the adapter's implementation via an environment variable.

```
apps/
  web/                     → Next.js 15 (App Router) — dashboard + candidate portal + API
packages/
  db/                      → Drizzle ORM schema + migrations (Postgres)
  core/                    → business logic, organized by module (future services)
    jobs/                  → JD generation, job CRUD
    candidates/            → resume parsing, scoring, ranking
    screening/             → Round-1 screening (async text/voice)
    scheduling/            → slots, bookings, calendar sync
    interviews/            → AI interview engine (Round 2)
    analytics/             → pipeline funnel, reports
  adapters/                → ALL external dependencies behind interfaces
    llm/                   → generate(), extract(), score(), embed()
    storage/               → put(), get(), signedUrl()        (S3-compatible API)
    queue/                 → enqueue(), worker()               (pg-boss now, BullMQ later)
    email/                 → send()
    telephony/             → call(), transcribe()              (stub in MVP)
  shared/                  → types, zod schemas, auth helpers
```

**Rule that makes migration painless:** `core/*` modules never import each other's internals and never call vendors directly — only through `adapters/*` and the DB. When you outgrow the monolith, each `core/` folder becomes a service with the same interface, and each adapter gets a production implementation.

---

## 2. MVP Tech Stack (all free tiers)

| Layer | MVP (free) | Scale target (from HireAI-Architecture.md) | Migration effort |
|---|---|---|---|
| Frontend + API | Next.js 15 + TypeScript + Tailwind + shadcn/ui on **Vercel free** | Same frontend; API extracted to NestJS/Fastify pods | Low — `core/` is framework-agnostic |
| Database | **Supabase free** (Postgres 500MB + pgvector) | RDS / Supabase Pro Postgres | None — same engine, same Drizzle migrations |
| ORM | Drizzle (SQL-first, no runtime engine) | Same | None |
| Auth | **Supabase Auth** (email + Google OAuth, free) | Same, or JWT/OAuth2 service | None until enterprise SSO |
| Vector search | **pgvector** in same Postgres | pgvector on bigger instance (Pinecone only if >10M vectors) | None |
| File storage | **Supabase Storage** (1GB free, S3-compatible) | AWS S3 | Env-var swap — same S3 API in adapter |
| Background jobs | **pg-boss** (queue inside Postgres — zero extra infra) | Redis + BullMQ | Adapter swap; job payloads identical |
| Cache | None needed at this scale (Postgres is fast enough) | Redis / Upstash | Add later behind adapter |
| Full-text search | Postgres `tsvector` | Elasticsearch/Typesense if needed | Deferred; likely never needed |
| Realtime (live transcript) | Supabase Realtime channels | Socket.io / managed WS | Adapter swap |
| Email | Resend free (3k/mo) or Brevo free (300/day) | SendGrid/SES | Adapter swap |
| Monitoring | Sentry free + Vercel analytics | Sentry + Datadog + PostHog | Additive |
| Billing | Skip in MVP (manual invoices) | Stripe | Additive module |

**Why Supabase as the spine:** one free account gives Postgres + pgvector + Auth + Storage + Realtime — the entire data layer of the target architecture, with a direct paid upgrade path and no vendor rewrite (it's plain Postgres and S3 APIs underneath).

---

## 3. Free AI Stack (the important part)

All AI calls go through **one adapter**: `adapters/llm` exposing `generate()`, `extractStructured()`, `scoreAgainstRubric()`, `embed()`. Provider chosen by env var, with automatic fallback chain when a free tier rate-limits.

| Capability | Free option (MVP) | Notes / limits | Scale swap |
|---|---|---|---|
| JD generation, scoring, question gen | **Google Gemini 2.5 Flash** free tier | ~1,500 req/day free — plenty for 1k users | Claude Sonnet/Opus via API key swap |
| Fallback LLM | **Groq** (Llama 3.3 70B) free tier | Very fast; good for scoring/extraction | Same |
| Structured resume extraction | Gemini Flash with JSON schema output | pdf-parse / mammoth for text extraction first (free, local) | Claude with tool-use |
| Embeddings (candidate↔JD matching) | **Gemini `text-embedding-004`** (free) or local `bge-small` via transformers.js | Stored in pgvector | OpenAI/Voyage embeddings |
| Speech-to-text | **Groq Whisper large-v3** (free tier) | For uploaded audio answers | OpenAI Whisper / Deepgram |
| Text-to-speech | **Edge-TTS** (free, unofficial MS voices) or browser SpeechSynthesis | Good enough for async interviews | ElevenLabs / OpenAI TTS |
| Dev-time AI | **Claude Code** | Builds/refactors the codebase itself | — |

**Rate-limit strategy:** every LLM call goes through the queue with retry + provider fallback (Gemini → Groq → defer). At 1,000 users this stays inside free tiers because the heavy AI work (parsing, scoring) is per-candidate, batched, and cached (never re-score an unchanged resume — hash the resume+JD pair).

---

## 4. The One Big MVP Substitution: Voice → Async

Twilio outbound IVR calls are the most expensive, most complex, least-free part of the target architecture. For the MVP, **replace live phone screening with an async web-based screening**, which keeps the exact same data model (`screening_calls` table: transcript, scores, recording URL):

- **Round 1 (MVP):** Candidate gets an email/WhatsApp link → opens a mobile web page → answers 5–8 questions by **recording audio in the browser** (MediaRecorder API, free) → uploads to storage → queue job runs Groq Whisper STT → LLM scores against rubric → auto-advance past threshold.
- **Round 2 (MVP):** Same page, but conversational: TTS asks a question (Edge-TTS/browser), candidate answers by voice, STT + LLM generate an adaptive follow-up. This is the full "AI interview" experience with zero telephony cost.
- **Scale:** implement `adapters/telephony` with Twilio + streaming STT — the screening module's inputs/outputs (transcript in, scores out) don't change at all.

This is the design decision that makes a free MVP possible without repainting yourself into a corner.

---

## 5. Data Model (multi-tenant from day 1)

Use the schema already defined in `HireAI-Architecture.md` §5 almost verbatim — it's good. Non-negotiables to bake in now because they're brutal to retrofit:

1. **`org_id` on every table** + Postgres **Row-Level Security** (Supabase makes this native). Multi-tenancy is the #1 thing you cannot bolt on later.
2. **UUIDs for all primary keys** (no sequential int leaks, safe for future sharding).
3. **Status fields as explicit state machines** (`candidate.status`: `uploaded → parsed → scored → screening_invited → screening_done → interview_scheduled → interview_done → shortlisted / rejected`). The whole pipeline automation hangs off these transitions.
4. **`events` append-only table** (org_id, entity, entity_id, event_type, payload, created_at) — this gives you analytics, audit trail, and debugging for free, and becomes your event bus when you later move to real queues/services.
5. **All AI outputs stored as `jsonb` with the prompt version** (`ai_scores: {version: "v3", ...}`) so you can re-score when prompts improve and audit for EEOC/bias compliance.

---

## 6. Phase Plan (each phase ships usable value)

**Phase 0 — Foundation (week 1)**
Monorepo (pnpm + Turborepo), Next.js app, Supabase project, Drizzle schema + RLS, Supabase Auth (email + Google), org/user onboarding, CI via GitHub Actions (free), Sentry. Deploy to Vercel on day 1 — deploy continuously from here.

**Phase 1 — Jobs + JD Generation (week 2)**
Job CRUD, LLM adapter with Gemini/Groq fallback, JD generator with templates, skills extraction into structured fields. *First demoable value.*

**Phase 2 — Resume Pipeline (weeks 3–4)**
Bulk upload (drag-drop → Supabase Storage), pg-boss queue, text extraction (pdf-parse/mammoth), LLM structured extraction, embedding + pgvector similarity, weighted scoring vs JD, ranked candidate list with threshold filter. *This is the core product — most of the recommendation-engine work lives here.*

**Phase 3 — Async Screening, Round 1 (weeks 5–6)**
Candidate-facing mobile web page, browser audio recording, Whisper STT via Groq, rubric scoring, auto-advance on threshold, recruiter review UI with playback + transcript.

**Phase 4 — Scheduling (week 7)**
Recruiter availability slots, candidate self-serve booking page, Google Calendar API (free), email notifications via Resend. Skip Outlook until a customer asks.

**Phase 5 — AI Interview, Round 2 (weeks 8–9)**
Conversational async interview: TTS question → voice answer → adaptive follow-up, dimensional scoring, report generation. Reuses everything from Phase 3.

**Phase 6 — Pipeline Dashboard + Handoff (week 10)**
Funnel view per job, candidate comparison, one-click Round-3 scheduling, org analytics off the `events` table. Then: beta users.

**Deferred until revenue:** Twilio live IVR, Outlook/MS Graph, Stripe billing, Elasticsearch, Redis, SOC 2 tooling.

---

## 7. Migration Playbook (MVP → Scale, "starting tomorrow")

Triggers and the exact move for each — no rewrites, because of the adapter boundaries:

| Pain signal | Move | Why it's cheap |
|---|---|---|
| Free LLM rate limits hit | Set `LLM_PROVIDER=anthropic`, add API key | Same adapter interface |
| Supabase 500MB DB full | Upgrade Supabase plan or `pg_dump` → RDS | Plain Postgres |
| Queue jobs backing up | Swap pg-boss impl → BullMQ + Upstash Redis in `adapters/queue` | Same job payloads |
| Storage >1GB | Point storage adapter at S3 | Same S3 API |
| Vercel function timeouts on long AI jobs | Move worker process to Railway/Fly.io container (queue consumer already runs standalone) | Worker is already a separate entrypoint |
| Need live phone screening (paying customers) | Implement `adapters/telephony` with Twilio; screening module unchanged | Transcript-in/scores-out contract |
| API load / team growth | Extract `core/candidates` (heaviest) into its own service first; `events` table becomes the integration point | Module boundaries = service boundaries |
| Enterprise deals | Add Stripe, SSO (SAML via WorkOS), SOC 2 logging on the `events` table | Additive |

**Monthly cost: $0 for the MVP phase.** First paid line items, in order, will be: LLM API (~$50–200), Supabase Pro ($25), a worker container (~$5–10).

---

## 8. Claude Code Working Method

- Keep this file + a `CLAUDE.md` (conventions: adapter rule, RLS rule, state-machine statuses) at repo root so every session has the architecture in context.
- Build phase by phase; each phase = one milestone with its own branch and deployed preview.
- Have Claude Code write integration tests for each `core/` module against a local Postgres (Docker) — these tests are your safety net for every adapter swap during migration.
- Use zod schemas in `packages/shared` as the single source of truth for API contracts — when modules become services, the contracts already exist.
""",
    },
    # add more documents here, same shape
]


print("Documents:", len(documents))
print("Characters:", len(documents[0]["text"]))

## Step 3 - Chunking

An 11,000 character document is too big to search inside, and too big to paste into every prompt. So we cut it into small pieces called **chunks** of about 700 characters.

We start a new chunk in two situations:

1. We hit a Markdown heading (`#`) - the author is telling us a new topic starts
2. The current chunk is full

Each chunk comes back as a dictionary with an `id`, a `title` and the `text`.

> **The bug from the live class was here.** The `chunks.append(...)` and `return chunks` lines were indented *inside* the `for` loop, so the function returned after reading the very first line and gave you 1 chunk. Look at the indentation of the last block below - it is outside the loop. Python never warns you about this; the code just runs and gives the wrong answer.

In [ ]:
def chunk_document(doc, max_size=700):
    """Cut one document into chunks. Returns a list of dictionaries."""

    chunks = []
    section = "Introduction"    # the heading we are currently under
    buffer = ""                 # text collected so far for the current chunk

    for line in doc["text"].split("\n"):

        is_heading = line.startswith("#")
        is_full = len(buffer) + len(line) > max_size

        # save the chunk we have collected, if there is one
        if buffer.strip() != "" and (is_heading or is_full):
            chunks.append({
                "id": doc["title"] + " #" + str(len(chunks)),
                "title": doc["title"] + " > " + section,
                "text": buffer.strip(),
            })
            buffer = ""

        if is_heading:
            section = line.replace("#", "").strip()
        else:
            buffer = buffer + line + "\n"

    # save whatever is left over.
    # NOTE: this block is OUTSIDE the for loop. That was the live class bug.
    if buffer.strip() != "":
        chunks.append({
            "id": doc["title"] + " #" + str(len(chunks)),
            "title": doc["title"] + " > " + section,
            "text": buffer.strip(),
        })

    return chunks

In [ ]:
# Chunk every document into one flat list
chunks = []
for doc in documents:
    chunks = chunks + chunk_document(doc)

print("Total chunks:", len(chunks))
print()

for c in chunks:
    print(len(c["text"]), "chars |", c["title"])

In [ ]:
# Always read a few of your own chunks before going further.
# Most retrieval problems are visible right here.

print(chunks[0]["text"])
print()
print("-" * 60)
print()
print(chunks[7]["text"])

## Step 4 - Embeddings

A computer cannot compare two sentences for **meaning**. It can compare numbers very well.

An **embedding** is a list of numbers representing the meaning of a text. The useful property:

> Two texts with a similar meaning get similar numbers, even when they share no words.

We send many texts in **one API call** rather than one call per text. That is faster, cheaper, and how you would do it in production.

In [ ]:
def embed_texts(texts):
    """Turn a list of texts into a list of embeddings, in one API call."""
    response = openai.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    return [item.embedding for item in response.data]


test = embed_texts(["How do I roll back a deployment?"])

print("An embedding is a list of", len(test[0]), "numbers")
print("First 8:", test[0][:8])

In [ ]:
import numpy as np

def similarity(a, b):
    """Cosine similarity. 1 = same meaning, 0 = unrelated."""
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


sentences = [
    "How do I roll back a deployment?",          # compare everything to this
    "What is the process to revert a release?",  # same meaning, no shared words
    "Undo the last deploy of the service.",      # related
    "How do I apply for annual leave?",          # different topic
    "The capital of France is Paris.",           # unrelated
]

vectors = embed_texts(sentences)     # all five in one call

for i in range(len(sentences)):
    print(round(similarity(vectors[0], vectors[i]), 3), "|", sentences[i])

Sentence 2 shares no important word with sentence 1 - "roll back" vs "revert", "deployment" vs "release" - and still scores high.

A keyword search would score that pair zero. This is the whole reason RAG works.

## Step 5 - Store the chunks in ChromaDB

ChromaDB is a vector database that runs inside the notebook. No server, no account.

It stores four things per chunk:

| Field | What it is | Why |
|---|---|---|
| `ids` | unique name | so you can update or delete it later |
| `embeddings` | the numbers | this is what the search compares |
| `documents` | the text | to read it and put it in the prompt |
| `metadatas` | extra info | to show the source, and to filter |

All four lists must be the same length. Because we build all four from the same `chunks` list in one place, **they cannot get out of sync** - which is exactly the bug that broke the live class.

In [ ]:
import chromadb

chroma_client = chromadb.Client()   # in memory. Use PersistentClient(path="./db") to keep it.

# delete first so this cell can be run again without an error
try:
    chroma_client.delete_collection("docs")
except:
    pass

collection = chroma_client.create_collection(
    name="docs",
    metadata={"hnsw:space": "cosine"}   # cosine similarity, same as Step 4
)

# All four lists come from the same source, so they always match.
collection.add(
    ids=[c["id"] for c in chunks],
    embeddings=embed_texts([c["text"] for c in chunks]),
    documents=[c["text"] for c in chunks],
    metadatas=[{"title": c["title"]} for c in chunks],
)

print("Stored", collection.count(), "chunks")

## Step 6 - Search

Embed the question, then ask ChromaDB for the closest chunks.

Two things to know:

- ChromaDB returns a **distance**, not a similarity. Smaller is closer, so we convert with `1 - distance`.
- It returns lists inside lists, because you can search several questions at once. We only send one, so we always take `[0]`.

The optional `where` argument filters on metadata **during** the search. One function handles both plain search and filtered search.

In [ ]:
def search(question, n=3, where=None):
    """Find the n chunks closest in meaning to the question."""

    results = collection.query(
        query_embeddings=embed_texts([question]),
        n_results=n,
        where=where,          # e.g. {"title": "..."} to search one section only
    )

    hits = []
    for i in range(len(results["ids"][0])):
        hits.append({
            "title": results["metadatas"][0][i]["title"],
            "text": results["documents"][0][i],
            "score": 1 - results["distances"][0][i],
        })
    return hits


for hit in search("Which database should the MVP use and why?"):
    print(round(hit["score"], 3), "|", hit["title"])
    print("   ", hit["text"][:120].replace("\n", " "), "...")
    print()

In [ ]:
# Notice: none of these questions use the same words as the document,
# and the right section still comes back.

for q in ["how do we handle phone calls in the MVP?",
          "what happens when the free AI tier runs out?",
          "what must we get right in the database from day one?"]:

    print("Q:", q)
    for hit in search(q, n=2):
        print("   ", round(hit["score"], 3), "|", hit["title"])
    print()

## Step 7 - Answer the question

Now put the retrieved chunks into a prompt.

Three rules in the prompt do most of the work:

1. **Only use the documents given.** Stops the model falling back on training memory.
2. **Cite the source number.** Lets the reader verify.
3. **Say "I don't know" when the documents do not cover it.**

Rule 3 is the most important line in this notebook. A model will not refuse unless you explicitly allow it to.

In [ ]:
SYSTEM_PROMPT = """You answer questions about internal technical documents.

Rules:
1. Answer ONLY using the documents below. Do not use any other knowledge.
2. Write the source number after each fact, like this: [1]
3. If the documents do not contain the answer, reply exactly:
   "I don't know based on the documents provided."
4. Be short and clear."""


def ask(question, n=3):
    """Full RAG: search, build the prompt, get the answer."""

    hits = search(question, n=n)

    # number the sources so the model can cite them
    sources = []
    for i, hit in enumerate(hits, start=1):
        sources.append("[" + str(i) + "] " + hit["title"] + "\n" + hit["text"])

    user_prompt = "DOCUMENTS:\n" + "\n\n".join(sources) + "\n\nQUESTION: " + question

    response = openai.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0,      # same answer every time
    )

    answer = response.choices[0].message.content

    print(answer)
    print("\nSOURCES:")
    for i, hit in enumerate(hits, start=1):
        print(" ", "[" + str(i) + "]", hit["title"], "|", round(hit["score"], 3))
    print("\nTokens:", response.usage.total_tokens)

    return answer

In [ ]:
ask("Which database should the MVP use, and why was it chosen?")

In [ ]:
ask("How is voice screening handled in the MVP instead of Twilio calls?")

### The test that matters most

Ask something the document does not cover. A good RAG system refuses. A bad one invents.

This single test decides whether you can put the system in front of a real user.

In [ ]:
ask("What is our company's deploy window for production?")

**Try it:** delete rule 3 from `SYSTEM_PROMPT`, re-run that cell and this one. The model will invent a deploy window.

## Step 8 - Filtering

`where` narrows the search to matching metadata **before** ranking, so other chunks are never considered.

In a real product this is how access control works: store `user_id` in the metadata and filter on it, so a user physically cannot retrieve someone else's documents.

In [ ]:
section = "HireAI Architecture Plan > 2. MVP Tech Stack (all free tiers)"

print("Whole document:")
for hit in search("what do we use for storage?", n=3):
    print(" ", round(hit["score"], 3), "|", hit["title"])

print("\nOnly the tech stack section:")
for hit in search("what do we use for storage?", n=2, where={"title": section}):
    print(" ", round(hit["score"], 3), "|", hit["title"])

## Step 9 - Chat

Type `exit` to stop.

In [ ]:
while True:
    user_input = input("You: ")
    if "exit" in user_input.lower():
        break
    print()
    ask(user_input)
    print()

## Step 10 - Use your own documents

Replace the `documents` list in Step 2, then re-run Step 3 onwards. Nothing else changes.

In [ ]:
# --- text file (upload it with the folder icon in Colab) ---
# with open("/content/my_file.txt") as f:
#     documents = [{"title": "My File", "text": f.read()}]

# --- PDF ---
# !pip install -q pypdf
# from pypdf import PdfReader
# reader = PdfReader("/content/my_report.pdf")
# text = "\n\n".join(page.extract_text() for page in reader.pages)
# documents = [{"title": "My Report", "text": text}]
#
# A PDF has no # headings, so every chunk will be under "Introduction".
# That is fine - it still cuts by size.

# --- several documents ---
# documents = [
#     {"title": "Architecture Plan", "text": text_1},
#     {"title": "Security Policy",   "text": text_2},
# ]

print("Uncomment whichever you need, then re-run from Step 3")

## Summary

The whole system, in the order it runs:

| Step | Code | What it does |
|---|---|---|
| 2 | `documents` | a list of `{"title", "text"}` dictionaries |
| 3 | `chunk_document()` | cuts each document into `{"id", "title", "text"}` chunks |
| 4 | `embed_texts()` | turns a list of texts into a list of number-vectors |
| 5 | `collection.add()` | stores ids, embeddings, text and metadata together |
| 6 | `search()` | embeds the question, returns the closest chunks |
| 7 | `ask()` | puts those chunks in a prompt, gets a cited answer |

### Five things to remember

1. **RAG = search + a model that writes.** Most RAG problems are search problems.
2. **One dictionary per chunk, not parallel lists.** Lists that must stay the same length will eventually not.
3. **Embeddings match meaning, not words.** That is why "revert a release" finds "rollback".
4. **Always allow "I don't know".** Without that rule the model invents.
5. **Never put an API key in a notebook you share.**

### When an answer is wrong, it is one of three things

| Cause | How to check | Fix |
|---|---|---|
| The right text was never in a chunk | print your chunks and read them | chunking |
| It was in a chunk but search missed it | run `search()` on its own | raise `n`, better chunks |
| Search found it, answer still wrong | the right chunk IS in the sources list | the prompt |

**Run `search()` on its own first.** Five seconds, and it tells you whether the problem is retrieval or the model. Most people skip this and spend an hour rewriting the prompt when the chunking was wrong.

---

### What a production version adds

You now have the core. Real systems add these on top, in roughly this order:

| Add | Why |
|---|---|
| Keyword search (BM25) alongside embeddings | embeddings are weak on exact codes and IDs like `ERR_5012` |
| A reranker | a second, more accurate pass over the top 20 candidates |
| A saved test set of questions + expected sources | so you can prove a change helped instead of guessing |
| `PersistentClient` and re-indexing only changed files | not rebuilding everything for a one-page edit |
| Logging the retrieved chunk ids per question | so "it gave a wrong answer yesterday" is debuggable |

None of them change the six boxes at the top. They make each box better.

## Homework

1. **Break it on purpose.** Indent the last `if` block in `chunk_document()` into the `for` loop. Re-run Step 3. You should get 1 chunk. Then fix it.
2. Change `max_size` from 700 to 150, re-run Steps 3 and 5, then ask a question. Better or worse? Why?
3. Change `n=3` to `n=1` in `ask()` and try the data model question. What is missing?
4. Delete rule 3 from `SYSTEM_PROMPT` and ask about the deploy window again.
5. Add a second document to the list and ask a question that needs both.

---
---

# Part 2 - Types of RAG

> **Before you run this part:** Step 9 above is a `while True` chat loop that waits for your typing. Skip that one cell (or type `exit` in it) - everything below only needs Steps 1-8 to have run. It reuses your existing `chunks`, `collection`, `search()` and `SYSTEM_PROMPT`.

Everything above is **one** kind of RAG. It is the simplest kind, and it is the right starting point for every project.

But it fails in predictable ways, and each "type of RAG" you read about online is just a fix for one of those failures. That is the honest way to think about this:

> **A "type of RAG" is not a different system. It is one extra box added to the six boxes you already built.**

Here is the whole map. Read the right-hand column - it tells you *why each type exists*.

| Type | The one sentence version | Fixes this failure |
|---|---|---|
| **1. Naive / Basic RAG** | embed → search → stuff into prompt | nothing, it *is* the baseline |
| **2. Hybrid RAG** | vector search + keyword search, results merged | exact codes, IDs, names, rare words |
| **3. Reranking RAG** | fetch 20 cheaply, then reorder to the best 3 | the right chunk was at position 9, you only sent 3 |
| **4. Query rewriting / Multi-query / HyDE** | rewrite the question before searching | vague questions, follow-ups, wording mismatch |
| **5. Self-query / filtered RAG** | model writes a metadata filter, then search | "only 2024 policies", "only my documents" |
| **6. Parent-document RAG (small-to-big)** | search small chunks, send the big surrounding section | the answer got cut in half by chunking |
| **7. Agentic RAG (incl. Corrective / Self-RAG)** | decide whether to search, grade what came back, retry | one search was not enough, or was not needed |
| **8. Graph RAG** | build a graph of entities and relationships, walk it | "how is A connected to B", multi-hop questions |
| **9. Multimodal RAG** | retrieve images, tables and charts too | the answer lives in a diagram, not in a sentence |
| **10. Structured / SQL RAG (text-to-SQL)** | model writes a query against a database | "how many", "average", "top 5" - numbers |

Where they plug in:

```
   question
      |
      |   <-- 4. rewrite the question      (multi-query, HyDE)
      |   <-- 5. add a metadata filter     (self-query)
      v
   SEARCH  <-- 2. also do keyword search   (hybrid)
      |     <-- 8/10. or search a graph / a SQL database
      v
   top 20
      |   <-- 3. rerank down to the best 3
      |   <-- 6. expand each hit to its full section
      v
   PROMPT
      |
      v
   ANSWER  <-- 7. grade it. Not good enough? go back and search again
```

None of this is exotic. In the next cells you build types 2, 3, 4, 5, 6 and 7 with the functions you already have.

## Setup for Part 2

Three tiny helpers, used by everything below. Nothing new conceptually - we are just splitting "call the model" away from "print the answer" so we can reuse it inside loops and inside the evaluation code.

- `chat()` - send a prompt, get text back
- `chat_json()` - send a prompt, get a **dictionary** back (used for grading, routing, reranking)
- `answer_from_hits()` - the same prompt as `ask()`, but it returns the answer instead of printing it

`chat_json()` uses OpenAI's JSON mode (`response_format={"type": "json_object"}`). That forces valid JSON, so `json.loads()` cannot crash on a stray sentence like "Sure, here is your JSON:".

In [ ]:
import json

def chat(prompt, system="You are a precise assistant.", temperature=0):
    """Send one prompt, get text back."""
    response = openai.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": prompt}],
        temperature=temperature,
    )
    return response.choices[0].message.content


def chat_json(prompt, system="Reply with JSON only."):
    """Send one prompt, get a Python dictionary back."""
    response = openai.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": prompt}],
        temperature=0,
        response_format={"type": "json_object"},   # forces valid JSON
    )
    return json.loads(response.choices[0].message.content)


def answer_from_hits(question, hits):
    """Same prompt as ask(), but returns the answer instead of printing it."""
    sources = []
    for i, hit in enumerate(hits, start=1):
        sources.append("[" + str(i) + "] " + hit["title"] + "\n" + hit["text"])

    user_prompt = "DOCUMENTS:\n" + "\n\n".join(sources) + "\n\nQUESTION: " + question
    return chat(user_prompt, system=SYSTEM_PROMPT)


print("Helpers ready")

## Type 1 - Naive RAG (what you already built)

```
question -> embed -> search -> top 3 chunks -> prompt -> answer
```

**Where it is genuinely used in industry:** internal wikis and handbooks, product documentation bots, onboarding assistants, FAQ bots, and basically every proof-of-concept in every company. If your document set is small and the questions are phrased normally, this is 80% of the value for 5% of the work. Start here, always.

**Where it breaks - and these are the exact reasons the other 9 types exist:**

| The question | What goes wrong |
|---|---|
| "What does pg-boss do?" | a bare product name has almost no meaning for an embedding model → **Hybrid** |
| "What is the migration effort for the database row?" | the right chunk ranked 9th, you only looked at 3 → **Reranking** |
| "And what about the second one?" | the follow-up alone means nothing → **Query rewriting** |
| "What changed in the 2024 version?" | you cannot express "2024 only" in a vector → **Self-query** |
| a table split across a chunk boundary | half the answer is missing → **Parent-document** |
| "Compare the MVP stack with the scale stack" | needs two searches, not one → **Agentic** |
| "How many free-tier services do we depend on?" | counting is not retrieval → **SQL RAG** |

Run the cell below to *see* the first failure, rather than take my word for it.

In [ ]:
# A bare product name. Vector search is weak on these.
for hit in search("pg-boss", n=3):
    print(round(hit["score"], 3), "|", hit["title"])

print()
print("Look at the scores. They are low and the sections may be wrong.")
print("The word is IN the document, but 'pg-boss' as a vector means almost nothing.")

## Type 2 - Hybrid RAG (vector + keyword)

Two searches, two different strengths:

| Search | Good at | Bad at |
|---|---|---|
| **Vector** (what we have) | meaning: "revert a release" finds "rollback" | exact tokens: `ERR_5012`, `pg-boss`, `Form 16`, a person's name |
| **Keyword** (BM25) | exact tokens, rare words, IDs, codes | synonyms, paraphrases |

Hybrid runs both and merges the two ranked lists. The standard merge is **Reciprocal Rank Fusion (RRF)** - a formula so simple it is almost funny:

```
score(chunk) = sum over each list of   1 / (60 + rank in that list)
```

A chunk that appears near the top of *either* list gets a good score. A chunk near the top of *both* wins. The `60` is a smoothing constant from the original paper; nobody tunes it.

```
            +--> vector search --> [C3, C7, C1]  --+
question ---|                                      +--> RRF merge --> final top 3
            +--> keyword search -> [C7, C9, C3]  --+
```

**Where it is used in industry**
- **Customer support**: users paste error codes (`0x80070005`) and product SKUs
- **E-commerce search**: brand and model numbers must match exactly ("Samsung QN90B")
- **Legal / compliance**: statute and clause numbers ("Section 80C", "GDPR Art. 17")
- **Healthcare**: drug names, ICD-10 codes
- **Banking**: form names, policy numbers, IFSC codes

Rule of thumb: **if your users type identifiers, you need hybrid.** In production people use `rank_bm25`, Elasticsearch/OpenSearch, or a database with built-in hybrid (Weaviate, Qdrant, pgvector + `tsvector`). Below we write a 15-line keyword scorer so you can see there is no magic in it.

In [ ]:
import re
from collections import Counter

def keyword_search(question, n=3):
    """Simple keyword scoring: how many of the question's words appear in the chunk."""

    words = re.findall(r"[a-z0-9\-]+", question.lower())   # split into simple tokens
    words = [w for w in words if len(w) > 2]               # drop "is", "a", "of"

    scored = []
    for c in chunks:
        text = c["text"].lower()
        hits = sum(text.count(w) for w in words)           # raw count of matches
        rare = sum(1 for w in words if w in text)          # how many distinct words matched
        if hits > 0:
            scored.append({"title": c["title"], "text": c["text"], "score": rare * 10 + hits})

    scored.sort(key=lambda x: -x["score"])                 # highest score first
    return scored[:n]


def hybrid_search(question, n=3, pool=8):
    """Run both searches, merge with Reciprocal Rank Fusion."""

    vector_hits = search(question, n=pool)
    keyword_hits = keyword_search(question, n=pool)

    fused = {}
    for ranked_list in [vector_hits, keyword_hits]:
        for rank, hit in enumerate(ranked_list):
            key = hit["text"]                              # the chunk text identifies the chunk
            if key not in fused:
                fused[key] = {"title": hit["title"], "text": hit["text"], "score": 0}
            fused[key]["score"] += 1 / (60 + rank)         # <-- the whole RRF formula

    merged = sorted(fused.values(), key=lambda x: -x["score"])
    return merged[:n]


question = "What does pg-boss do?"

print("VECTOR ONLY")
for hit in search(question, n=3):
    print("  ", hit["title"])

print("\nKEYWORD ONLY")
for hit in keyword_search(question, n=3):
    print("  ", hit["title"])

print("\nHYBRID (merged)")
for hit in hybrid_search(question, n=3):
    print("  ", hit["title"])

Try a few of your own queries in that cell. The pattern you should see:

- normal sentences ("how do we handle phone calls") - vector wins, keyword is noisy
- bare identifiers ("pg-boss", "pgvector", "Drizzle") - keyword wins
- hybrid is never the worst of the two

That last point is why hybrid is the default in most production search systems: it removes an entire class of embarrassing failure for very little cost. No extra LLM call, no extra latency worth mentioning.

## Type 3 - Reranking RAG (retrieve wide, then narrow)

Vector search is fast but rough. It compares the question to each chunk *independently*, using vectors computed before your question existed.

A **reranker** looks at the question and each candidate chunk **together** and asks "does this actually answer it?". That is far more accurate, but far too slow to run over 50,000 chunks. So:

```
question -> vector search top 20   (cheap, rough)
         -> rerank those 20        (expensive, accurate)
         -> keep the best 3        -> prompt
```

**This is usually the single biggest quality jump per hour of work.** The reason is subtle but important: retrieving 20 and reranking to 3 gives the model a *better* 3 than retrieving 3 directly - while keeping the prompt exactly the same size and cost.

**Where it is used in industry**
- Basically every serious RAG product: Cohere Rerank, Voyage rerank, Jina, or an open cross-encoder (`bge-reranker`, `ms-marco-MiniLM`) run locally
- Search inside SaaS products, where the top 3 results are all the user sees
- Anywhere the corpus is big enough that "close in meaning" returns 20 plausible chunks

**The cost** is latency and money: a real reranker adds ~100-300 ms; the LLM-as-reranker below adds one extra model call. In production use a dedicated reranker model, not an LLM - it is 10x cheaper and faster. We use the LLM here because it needs no new API key.

In [ ]:
def rerank(question, hits, keep=3):
    """Ask the model to score each candidate chunk 0-10 for usefulness."""

    numbered = []
    for i, hit in enumerate(hits):
        numbered.append("[" + str(i) + "] " + hit["text"][:500])   # trim to keep it cheap

    prompt = (
        "Question: " + question + "\n\n"
        "Passages:\n" + "\n\n".join(numbered) + "\n\n"
        'Score how useful each passage is for answering the question, 0-10.\n'
        'Reply as JSON: {"scores": [{"index": 0, "score": 7}, ...]}'
    )

    result = chat_json(prompt)

    for item in result["scores"]:
        i = item["index"]
        if 0 <= i < len(hits):
            hits[i]["rerank_score"] = item["score"]

    for hit in hits:
        hit.setdefault("rerank_score", 0)       # safety net if the model skipped one

    ranked = sorted(hits, key=lambda h: -h["rerank_score"])
    return ranked[:keep]


question = "What must we get right in the database from day one?"

wide = search(question, n=8)          # cast a wide net
print("BEFORE reranking (vector order):")
for i, hit in enumerate(wide):
    print("  ", i, round(hit["score"], 3), "|", hit["title"])

best = rerank(question, wide, keep=3)
print("\nAFTER reranking (best 3):")
for hit in best:
    print("  ", hit["rerank_score"], "|", hit["title"])

print("\nANSWER FROM THE RERANKED 3:")
print(answer_from_hits(question, best))

## Type 4 - Query rewriting: Multi-query and HyDE

The user's words and the document's words are often different, and users type short, lazy, ambiguous questions. Fix the question *before* searching.

Three variants you should know by name:

| Variant | What it does | Best for |
|---|---|---|
| **Rewrite / condense** | uses the chat history to turn "and the second one?" into a full standalone question | any chatbot with follow-ups |
| **Multi-query** | generates 3 different phrasings, searches all of them, merges | vague or broad questions |
| **HyDE** | asks the model to *invent* a fake ideal answer, then embeds **that** instead of the question | when questions and documents are written very differently |

HyDE (Hypothetical Document Embeddings) sounds strange but the logic is clean: a made-up answer looks a lot more like the real document than a question does. Questions and answers are written in different styles, and you are comparing against answers.

```
"what about storage?"  ->  ["which storage service does the MVP use?",
                            "how are files and resumes stored?",
                            "what is the storage adapter in the MVP stack?"]
                                    |
                        search all three, merge with RRF
```

**Where it is used in industry**
- Every multi-turn support chatbot (condensing is mandatory - without it turn 2 retrieves garbage)
- Enterprise search where users type 3-word queries
- Research and legal assistants, where one question genuinely needs several searches

**The cost:** one extra LLM call plus N searches. Multi-query roughly triples your retrieval cost, so use it where quality matters more than latency.

In [ ]:
def multi_query_search(question, n=3, variants=3):
    """Generate several phrasings, search each, merge with RRF."""

    prompt = (
        "Rewrite this question in " + str(variants) + " different ways that would each "
        "match different wording in a technical document. Keep the meaning identical.\n\n"
        "Question: " + question + "\n\n"
        'Reply as JSON: {"queries": ["...", "..."]}'
    )
    queries = chat_json(prompt)["queries"]
    queries = [question] + queries            # always keep the original too

    fused = {}
    for q in queries:
        for rank, hit in enumerate(search(q, n=5)):
            key = hit["text"]
            if key not in fused:
                fused[key] = {"title": hit["title"], "text": hit["text"], "score": 0}
            fused[key]["score"] += 1 / (60 + rank)

    print("Searched with:")
    for q in queries:
        print("   -", q)

    return sorted(fused.values(), key=lambda x: -x["score"])[:n]


def hyde_search(question, n=3):
    """HyDE: write a fake answer, then search using that instead of the question."""

    fake_answer = chat(
        "Write 3 sentences that would plausibly appear in a technical architecture "
        "document answering this question. Invent details if needed.\n\n" + question
    )
    print("Fake answer used as the search query:\n  ", fake_answer[:200].replace("\n", " "), "...\n")
    return search(fake_answer, n=n)


vague = "what about storage?"

print("PLAIN SEARCH")
for hit in search(vague, n=3):
    print("  ", round(hit["score"], 3), "|", hit["title"])

print("\nMULTI-QUERY")
for hit in multi_query_search(vague, n=3):
    print("  ", hit["title"])

print("\nHyDE")
for hit in hyde_search(vague, n=3):
    print("  ", round(hit["score"], 3), "|", hit["title"])

## Type 5 - Self-query RAG (the model writes the filter)

Step 8 showed manual filtering: you passed `where={"title": section}` yourself. **Self-query** means the model reads the question and decides the filter.

```
"what does the tech stack section say about auth?"
        |
        v
   model returns: {"section": "... > 2. MVP Tech Stack (all free tiers)"}
        |
        v
   search(question, where={"title": that section})
```

**Where it is used in industry**
- **Multi-tenant SaaS**: `where={"org_id": current_user.org_id}` - though note that tenant filters must be set by *your code*, never by the model
- **Time-bounded questions**: "what changed in the 2024 policy" → `where={"year": 2024}`
- **Document-type routing**: "in the contract, not the emails" → `where={"doc_type": "contract"}`
- **Large corpora**: filtering first makes the vector search both faster and more accurate

**The one rule you must not break:** security filters (user id, tenant id, permissions) are added by your code, not chosen by the LLM. Self-query is a *convenience* feature for narrowing topics. Treat a model-chosen filter as a suggestion; treat a permission filter as law.

In [ ]:
# The sections available to filter on
all_sections = sorted({c["title"] for c in chunks})

def self_query(question, n=3):
    """Let the model pick a section to search inside, or none at all."""

    prompt = (
        "Available document sections:\n" + "\n".join("- " + s for s in all_sections) + "\n\n"
        "Question: " + question + "\n\n"
        "If ONE section clearly contains the answer, return it exactly. Otherwise return null.\n"
        'Reply as JSON: {"section": "exact section name or null"}'
    )

    section = chat_json(prompt).get("section")

    if section in all_sections:
        print("Filtered to:", section)
        return search(question, n=n, where={"title": section})

    print("No filter - searching everything")
    return search(question, n=n)


for hit in self_query("in the tech stack section, what is used for auth?"):
    print("  ", round(hit["score"], 3), "|", hit["title"])

print()
for hit in self_query("what is this whole document about?"):
    print("  ", round(hit["score"], 3), "|", hit["title"])

## Type 6 - Parent-document RAG (small-to-big)

There is a real conflict in chunking:

- **Small chunks search better.** Less noise per chunk, so the vector is about one topic.
- **Big chunks answer better.** The model needs surrounding context, definitions, and the rest of the table.

Small-to-big gets both: **search the small chunk, then send the model its neighbours too.**

```
   search finds:              send to the model:

   +--------+                 +--------+
   | chunk 6|                 | chunk 5|   <- context before
   +--------+       ==>       | chunk 6|   <- the actual match
                              | chunk 7|   <- context after
                              +--------+
```

The full version stores small "child" chunks for searching and a link to the big "parent" document, then retrieves the parent. Our version is the cheap approximation: expand to the chunks either side.

**Where it is used in industry**
- **Legal**: a clause is meaningless without the definitions and the clause before it
- **Technical manuals and SOPs**: step 4 of a procedure needs steps 3 and 5
- **Financial filings**: a number needs its table header and footnote
- **Code assistants**: the matched function needs its imports and the class around it

**The cost:** a bigger prompt - typically 2-3x the tokens. Which is exactly why reranking (type 3) pairs so well with it: rerank down to 2 hits, then expand each one.

In [ ]:
def expand_hits(hits, window=1):
    """Replace each hit with itself plus its neighbouring chunks."""

    index_of = {c["text"]: i for i, c in enumerate(chunks)}   # text -> position in chunks

    expanded = []
    used = set()

    for hit in hits:
        i = index_of.get(hit["text"])
        if i is None:
            expanded.append(hit)
            continue

        start = max(0, i - window)
        end = min(len(chunks), i + window + 1)

        if (start, end) in used:        # do not send the same block twice
            continue
        used.add((start, end))

        block = "\n\n".join(chunks[j]["text"] for j in range(start, end))
        expanded.append({"title": hit["title"] + "  (+context)", "text": block, "score": hit["score"]})

    return expanded


question = "What must we get right in the database from day one?"

small = search(question, n=2)
big = expand_hits(small, window=1)

print("Small chunks: ", [len(h["text"]) for h in small], "chars")
print("With context: ", [len(h["text"]) for h in big], "chars")
print()
print("ANSWER WITH CONTEXT EXPANSION:")
print(answer_from_hits(question, big))

## Type 7 - Agentic RAG (decide, grade, retry)

So far the pipeline is fixed: always search once, always answer. An **agentic** RAG lets the model make decisions:

1. **Route** - does this question even need a search? ("hi" does not. "thanks" does not.)
2. **Search**
3. **Grade** - are these chunks actually relevant?
4. **Retry** - if not, rewrite the query and search again (max 2 times, or you get an infinite loop and an infinite bill)
5. **Answer**, or honestly refuse

```
question -> route ---- no search needed ----> answer directly
              |
           search
              |
            grade ---- bad ----> rewrite query --> search again (max 2 tries)
              |
             good
              v
           answer
```

Two named variants you will see in papers and job descriptions:

| Name | Idea |
|---|---|
| **Corrective RAG (CRAG)** | grade the retrieved chunks; if they are weak, fall back to a web search |
| **Self-RAG** | the model also grades **its own answer** for support and usefulness, and retries |

**Where it is used in industry**
- **Support agents** that must decide between the docs, the order database, and "escalate to a human"
- **Assistants over several sources** (wiki + tickets + code + the web)
- **Research and analyst tools**, where one question needs several searches in sequence

**The cost:** 2-5x the LLM calls and much harder to debug. Do not start here. Reach for agentic RAG only when you can show, with an eval run, that a single fixed search is what is limiting you.

In [ ]:
def agentic_rag(question, max_tries=2):
    """Route, search, grade, retry, answer."""

    # 1. route
    route = chat_json(
        "Does answering this need a search of internal technical documents? "
        "Small talk and greetings do not.\n\nQuestion: " + question + "\n\n"
        'Reply as JSON: {"needs_search": true or false}'
    )
    if not route["needs_search"]:
        print("Route: no search needed")
        return chat(question, system="Be brief and friendly.")

    query = question

    for attempt in range(1, max_tries + 1):
        # 2. search
        hits = hybrid_search(query, n=4)
        print("Attempt", attempt, "| query:", query)

        # 3. grade
        preview = "\n\n".join(h["text"][:300] for h in hits)
        grade = chat_json(
            "Question: " + question + "\n\nRetrieved passages:\n" + preview + "\n\n"
            "Do these passages contain enough information to answer the question?\n"
            'Reply as JSON: {"enough": true or false, "missing": "what is missing"}'
        )
        print("   grade:", "enough" if grade["enough"] else "not enough - " + grade.get("missing", "")[:70])

        if grade["enough"]:
            return answer_from_hits(question, hits)

        # 4. retry with a better query
        if attempt < max_tries:
            query = chat(
                "The search for this question missed. Missing: " + str(grade.get("missing", "")) +
                "\nWrite ONE better search query. Return only the query.\n\nQuestion: " + question
            )

    print("   giving up after", max_tries, "attempts")
    return "I don't know based on the documents provided."


print(agentic_rag("hi there, how are you?"))
print("\n" + "-" * 60 + "\n")
print(agentic_rag("Which queue does the MVP use and what does it move to at scale?"))
print("\n" + "-" * 60 + "\n")
print(agentic_rag("What is our production deploy window?"))

## Type 8 - Graph RAG (explanation only)

Vector search finds text that is *similar*. It cannot answer questions about *connections*.

> "Which teams depend on a service that depends on Supabase?"

That needs two hops. No single chunk contains the answer, so no amount of retrieval will find it.

**Graph RAG** extracts entities and relationships into a graph first:

```
(HireAI MVP) --[uses]--> (Supabase) --[provides]--> (Postgres + pgvector)
      |                                                   ^
   [uses]                                                 |
      v                                            [stores vectors in]
 (pg-boss) --[replaced by at scale]--> (BullMQ)            |
                                                    (resume search)
```

At query time you find the entities in the question, walk the edges, and put the *subgraph* in the prompt instead of raw text chunks.

**Where it is used in industry**
- **Pharma and biotech**: gene → protein → drug → trial chains
- **Fraud and AML**: account → device → address → account rings
- **Supply chain**: part → supplier → factory → region risk
- **Enterprise knowledge**: person → team → system → incident
- **Impact analysis in large codebases and IT estates**

**Why you should not start here:** building the graph costs one LLM call per chunk (expensive), the schema needs real thought, and it needs a graph store (Neo4j) or a library (LlamaIndex / Microsoft GraphRAG / LangChain's graph transformers). Most production systems that use graphs run them *alongside* normal vector RAG, not instead of it.

**The tell that you need it:** your users keep asking "how are these two things related?" and "what else is affected if X changes?" - and no single document ever contains the answer.

## Types 9 and 10, and the "just use a long context" question

**9. Multimodal RAG** - when the answer lives in a picture, not a paragraph. Architecture diagrams, screenshots, X-rays, engineering drawings, scanned invoices, product photos. Two common approaches:

- **Caption then index**: send each image to a vision model, store the description as text, retrieve the text, show the image to the model in the final prompt.
- **Native multimodal embeddings** (CLIP-style): embed images and text into the same vector space so a text query can retrieve an image directly.

Industry: insurance claim photos, medical imaging, e-commerce visual search, manufacturing defect reports, and any RAG over PDFs where the tables and charts carry the meaning.

**10. Structured / SQL RAG (text-to-SQL)** - "how many candidates applied last month?" is not a retrieval question, it is a `SELECT COUNT(*)`. The model writes SQL against your database, you run it (read-only user, always), and it explains the result. Industry: BI and analytics assistants, ops dashboards, finance reporting. Very often combined with document RAG in one agent: numbers come from SQL, policies come from documents.

**"Models have 1M-token context windows now. Is RAG dead?"** - a question you will be asked in an interview. The answer:

| | Long context | RAG |
|---|---|---|
| Cost per question | you pay for every token, every time | you pay for ~4 chunks |
| Latency | grows with context | flat |
| Accuracy | degrades in the middle of very long inputs | you control what goes in |
| Access control | all-or-nothing | per-chunk metadata filters |
| Citations | vague | exact chunk |
| Corpus size | capped by the window | unbounded |

Long context does not replace RAG; it makes RAG *more forgiving* - you can afford bigger chunks and more of them. The good answer is: **"long context reduces how perfect my chunking must be; it does not remove the need to choose what to send."**

## How to choose - the honest order

Do these in order. Stop as soon as your evaluation numbers are good enough (Part 3 tells you what "good enough" means).

| Order | Add | Cost | Typical quality gain |
|---|---|---|---|
| 1 | **Naive RAG done well** - read your chunks, allow "I don't know" | none | the baseline |
| 2 | **Hybrid search** | almost none | removes all identifier failures |
| 3 | **Reranking** (retrieve 20 → keep 3) | 1 cheap call | usually the biggest single jump |
| 4 | **Query rewriting / condensing** | 1 cheap call | essential the moment you have a chat UI |
| 5 | **Parent-document expansion** | more prompt tokens | fixes "half the answer" |
| 6 | **Metadata filters** (self-query) | ~none | required for multi-tenant or dated corpora |
| 7 | **Agentic loop** | 2-5x calls | only when one search is provably not enough |
| 8 | **Graph / multimodal / SQL** | high | only when the question type demands it |

The mistake to avoid: jumping to step 7 or 8 because it sounds impressive, while step 1 is still broken because nobody ever printed the chunks.

## Where each type shows up, by industry

| Industry | Typical use case | Types usually needed |
|---|---|---|
| **Software / DevTools** | docs bot, code assistant, incident search | naive + hybrid + reranking + parent-doc |
| **Customer support** | agent assist, deflection bot | hybrid (error codes) + reranking + agentic (order lookup) |
| **Banking / Insurance** | policy and product Q&A, claims triage | filtered (product, date) + hybrid + strict refusal + audit log |
| **Legal** | contract review, clause search, case research | parent-doc (definitions) + hybrid (citations) + reranking |
| **Healthcare / Pharma** | clinical guidelines, drug interactions, research | hybrid (codes) + graph + multimodal (imaging) + human review |
| **HR / People Ops** | leave, payroll, benefits questions | naive + filtered by country/entity + refusal |
| **E-commerce** | product Q&A, review summarisation, search | hybrid + multimodal + SQL (stock, price) |
| **Manufacturing / Field service** | equipment manuals, repair procedures | parent-doc + multimodal (diagrams) + offline-friendly index |
| **Consulting / Finance research** | report and filing analysis | multi-query + reranking + SQL + long context |
| **Telecom / Utilities** | plan and tariff questions, network faults | filtered + hybrid + graph (network topology) |

Two things are true in every row: **the refusal path matters more than the clever retrieval**, and **nobody ships without evaluation** - which is Part 3.

---
---

# Part 3 - RAG Evaluation

You now have eight ways to build the same system. So which one is better?

"It looked good when I tried it" is not an answer. It is how teams ship a bot that confidently invents a deploy window to a customer.

**Evaluation is just this:** a saved list of questions you know the answers to, and a script that scores your system against it. That is all. It is not a research activity - it is a test suite.

## The one idea that makes evaluation click

RAG has two halves, and they fail for different reasons. **Score them separately.**

```
                    RETRIEVAL                    GENERATION
                        |                             |
   question -> search -> right chunks? -> prompt -> right answer?
                        |                             |
              hit rate, MRR, recall        faithfulness, correctness
```

If retrieval is broken, no prompt on earth will save you. So **always measure retrieval first.** Most teams skip this, get a bad answer, and spend a week rewriting prompts when the chunk never made it into the search results.

## The four metrics that matter

| Metric | Question it answers | How it is measured |
|---|---|---|
| **Hit rate @ k** (retrieval) | did the right chunk appear in the top k? | code, free, no LLM |
| **MRR** (retrieval) | how *high up* was it? | code, free |
| **Faithfulness / groundedness** (generation) | is every claim supported by the retrieved text? | LLM judge |
| **Correctness** (generation) | does the answer match the known truth? | LLM judge |

Plus two that matter operationally: **refusal accuracy** (does it say "I don't know" when it should?) and **cost / latency per question**.

`faithfulness` is the anti-hallucination metric and the one to quote in an interview. Note it is different from correctness: an answer can be perfectly faithful to a chunk that was the wrong chunk.

## Step 11 - Build a test set

The test set is the most valuable file in a RAG project. It is also the most boring to make, which is why most teams do not have one.

Rules for a good test set:

1. **20-50 questions is plenty** to start. 6 is enough to learn the mechanics.
2. Write them from **real user questions**, not from reading the document. Questions people actually ask are shorter and messier than the ones you invent.
3. Include the boring ones, not just the clever ones.
4. **Always include questions the documents cannot answer.** If your test set is all answerable, you are not testing the refusal path - the single most dangerous failure mode.
5. Store an **expected source** and a **reference answer** per question.

In a real project, `expected_source` is the chunk id you labelled by hand. Here we use a simpler proxy: a keyword that must appear in the right chunk's text. It is less precise, and it means you do not have to hand-label 50 chunk ids to get started.

In [ ]:
# Each case: the question, a keyword that must appear in the correct chunk,
# a reference answer, and whether the system SHOULD refuse.
test_set = [
    {
        "question": "Which database should the MVP use, and why?",
        "expect_in_chunk": "Supabase",
        "reference": "Supabase free tier Postgres with pgvector, because it is the same engine as the scale target so migration needs no rewrite.",
        "should_refuse": False,
    },
    {
        "question": "What is used for the job queue in the MVP?",
        "expect_in_chunk": "pg-boss",
        "reference": "pg-boss running in Postgres in the MVP, moving to BullMQ at scale.",
        "should_refuse": False,
    },
    {
        "question": "How are phone calls handled in the MVP?",
        "expect_in_chunk": "telephony",
        "reference": "Telephony is stubbed in the MVP - screening is done as async text or voice instead of live calls.",
        "should_refuse": False,
    },
    {
        "question": "Why not use microservices and Kubernetes from day one?",
        "expect_in_chunk": "monolith",
        "reference": "Because it is unnecessary complexity for 1,000 users - a modular monolith with adapter boundaries is used instead so it can be split later.",
        "should_refuse": False,
    },
    {
        "question": "What is the frontend framework and where is it hosted?",
        "expect_in_chunk": "Next.js",
        "reference": "Next.js 15 with TypeScript and Tailwind, hosted on the Vercel free tier.",
        "should_refuse": False,
    },
    {
        "question": "What is our production deploy window?",
        "expect_in_chunk": None,          # nothing should match - the doc does not cover it
        "reference": "Not covered by the documents.",
        "should_refuse": True,
    },
    {
        "question": "How many employees does the company have?",
        "expect_in_chunk": None,
        "reference": "Not covered by the documents.",
        "should_refuse": True,
    },
]

print(len(test_set), "test cases |",
      sum(1 for t in test_set if t["should_refuse"]), "of them should be refused")

## Step 12 - Evaluate retrieval (free, fast, do this first)

Three metrics, all plain Python. No LLM, no cost, so you can run this every time you change chunking.

| Metric | Meaning | Reading it |
|---|---|---|
| **Hit rate @ k** | fraction of questions where the right chunk was in the top k | below 0.9 → fix retrieval before touching the prompt |
| **MRR** | 1/rank of the first correct chunk, averaged | 1.0 = always first, 0.5 = usually second |
| **Precision @ k** | fraction of returned chunks that were relevant | low means you are wasting prompt tokens |

MRR matters because of a real effect: models pay most attention to the start and end of a long prompt. A correct chunk sitting in position 5 of 5 is much weaker than the same chunk in position 1. Reranking exists to fix exactly this.

In [ ]:
def evaluate_retrieval(search_fn, cases, k=3):
    """Score any search function against the test set."""

    rows = []
    hits_total = mrr_total = precision_total = 0
    answerable = 0

    for case in cases:
        if case["expect_in_chunk"] is None:
            continue                                   # retrieval metrics need a right answer
        answerable += 1

        results = search_fn(case["question"], n=k)
        keyword = case["expect_in_chunk"].lower()

        found_ranks = [i for i, h in enumerate(results) if keyword in h["text"].lower()]

        hit = 1 if found_ranks else 0
        mrr = 1 / (found_ranks[0] + 1) if found_ranks else 0
        precision = len(found_ranks) / k

        hits_total += hit
        mrr_total += mrr
        precision_total += precision

        rows.append((("HIT " if hit else "MISS"),
                     round(mrr, 2),
                     case["question"][:46]))

    print(f"{'':6}{'MRR':<6}QUESTION")
    for r in rows:
        print(f"{r[0]:6}{r[1]:<6}{r[2]}")

    scores = {
        "hit_rate@" + str(k): round(hits_total / answerable, 2),
        "mrr": round(mrr_total / answerable, 2),
        "precision@" + str(k): round(precision_total / answerable, 2),
    }
    print("\n", scores)
    return scores


print("=== plain vector search ===")
vector_scores = evaluate_retrieval(search, test_set, k=3)

print("\n=== hybrid search ===")
hybrid_scores = evaluate_retrieval(hybrid_search, test_set, k=3)

Compare the two numbers. Hybrid usually wins on the `pg-boss` style question and ties elsewhere.

**This is the whole point of evaluation:** you just answered "is hybrid better for my documents?" with a number instead of an opinion, in about ten seconds and for zero dollars.

## Step 13 - Evaluate the answers (LLM as judge)

Retrieval is measurable with code. Answer quality is not - "is this answer faithful?" needs judgement. So we use a second model call as the judge.

Three judges, each answering exactly one question:

| Judge | Asks | Why it exists |
|---|---|---|
| `judge_faithfulness` | is every claim supported by the retrieved chunks? | catches hallucination |
| `judge_correctness` | does the answer match the reference? | catches wrong-but-confident answers |
| `check_refusal` | did it refuse when it should have? | catches the dangerous failure |

Notes on using LLM judges honestly, because this is where people fool themselves:

- **Give the judge a narrow question and a rubric.** "Rate this answer out of 10" produces noise. "Is every claim supported - yes or no" produces a usable signal.
- **temperature=0**, and JSON output, so the score is stable between runs.
- **A cheap model is fine as a judge.** Judging is easier than generating.
- **Spot-check the judge.** Read 10 of its verdicts yourself. If you disagree with it, fix its prompt before you trust any of its numbers.
- `check_refusal` needs no LLM at all - it is string matching. Prefer code over a judge whenever you can.

In [ ]:
def judge_faithfulness(answer, hits):
    """Is every claim in the answer supported by the retrieved chunks?"""
    context = "\n\n".join(h["text"] for h in hits)
    result = chat_json(
        "CONTEXT:\n" + context + "\n\nANSWER:\n" + answer + "\n\n"
        "Is every factual claim in the ANSWER supported by the CONTEXT? "
        "Ignore hedging and 'I don't know' statements.\n"
        'Reply as JSON: {"faithful": true or false, "unsupported": "the first unsupported claim, or none"}'
    )
    return result


def judge_correctness(question, answer, reference):
    """Does the answer match the known-correct reference?"""
    result = chat_json(
        "QUESTION: " + question + "\nREFERENCE ANSWER: " + reference + "\nGIVEN ANSWER: " + answer + "\n\n"
        "Is the GIVEN ANSWER factually consistent with the REFERENCE? Different wording is fine. "
        "Extra correct detail is fine. Missing the key fact, or contradicting it, is not.\n"
        'Reply as JSON: {"correct": true or false, "why": "one short sentence"}'
    )
    return result


def check_refusal(answer):
    """No LLM needed - just look for the refusal phrase."""
    text = answer.lower()
    return ("don't know" in text or "do not know" in text or "not contain" in text)


# quick sanity check of the judges on one question
case = test_set[0]
hits = search(case["question"], n=3)
answer = answer_from_hits(case["question"], hits)

print("ANSWER:", answer[:200], "...\n")
print("faithful  :", judge_faithfulness(answer, hits))
print("correct   :", judge_correctness(case["question"], answer, case["reference"]))
print("refused   :", check_refusal(answer))

## Step 14 - The full evaluation run

One function that runs the whole pipeline over the whole test set and prints a scorecard. This is the thing you re-run after every change.

It takes a `search_fn` argument, so you can score **any** of the retrieval types from Part 2 with the same harness - naive, hybrid, multi-query, reranked, whatever.

Cost warning: this makes roughly 3 model calls per question (answer + 2 judges). With 7 questions on `gpt-4o-mini` that is a fraction of a cent. With 500 questions and a big model, it is not - so run the free retrieval eval often and the full eval on a schedule.

In [ ]:
import time

def evaluate_rag(search_fn=search, cases=test_set, k=3, label="baseline"):
    """Run the full pipeline over the test set and score it."""

    print("=" * 74)
    print("EVALUATING:", label, "| k =", k)
    print("=" * 74)

    results = []
    started = time.time()

    for case in cases:
        hits = search_fn(case["question"], n=k)
        answer = answer_from_hits(case["question"], hits)

        refused = check_refusal(answer)

        if case["should_refuse"]:
            # only one thing matters here: did it refuse?
            row = {"q": case["question"], "retrieval": "-",
                   "faithful": "-", "correct": "PASS" if refused else "FAIL",
                   "refused": refused}
        else:
            keyword = case["expect_in_chunk"].lower()
            retrieved_ok = any(keyword in h["text"].lower() for h in hits)

            faith = judge_faithfulness(answer, hits)
            corr = judge_correctness(case["question"], answer, case["reference"])

            row = {"q": case["question"],
                   "retrieval": "PASS" if retrieved_ok else "FAIL",
                   "faithful": "PASS" if faith["faithful"] else "FAIL",
                   "correct": "PASS" if corr["correct"] else "FAIL",
                   "refused": refused,
                   "why": corr.get("why", "")}

        results.append(row)

    # ---- per question ----
    print(f"{'RETR':<6}{'FAITH':<7}{'CORR':<6}QUESTION")
    for r in results:
        print(f"{r['retrieval']:<6}{r['faithful']:<7}{r['correct']:<6}{r['q'][:44]}")

    # ---- totals ----
    answerable = [r for r in results if r["retrieval"] != "-"]
    refusals = [r for r in results if r["retrieval"] == "-"]

    def rate(rows, field):
        if not rows:
            return None
        return round(sum(1 for r in rows if r[field] == "PASS") / len(rows), 2)

    scores = {
        "retrieval_hit_rate": rate(answerable, "retrieval"),
        "faithfulness": rate(answerable, "faithful"),
        "correctness": rate(answerable, "correct"),
        "refusal_accuracy": rate(refusals, "correct"),
        "seconds": round(time.time() - started, 1),
    }

    print("\nSCORECARD:", scores)

    failures = [r for r in results if "FAIL" in (r["retrieval"], r["faithful"], r["correct"])]
    if failures:
        print("\nFAILURES to look at:")
        for r in failures:
            print("  -", r["q"][:60], "|", r.get("why", "")[:70])

    return scores


baseline = evaluate_rag(search, label="naive vector, k=3", k=3)

## Step 15 - Run an experiment (the part that makes you useful)

An evaluation number on its own is trivia. Evaluation becomes valuable when you use it to **compare two versions**.

The discipline is simple and it never changes:

1. Change **exactly one thing**
2. Re-run the same test set
3. Keep the change only if the number went up

Below we compare naive vector search against hybrid search, at the same `k`. Change one knob at a time and you always know what caused what.

Things worth comparing in your own project:

| Knob | Try |
|---|---|
| `max_size` in chunking | 300 vs 700 vs 1200 |
| `n` (chunks sent) | 1 vs 3 vs 6 |
| retrieval type | vector vs hybrid vs reranked vs multi-query |
| prompt | with and without rule 3 (the refusal rule) |
| model | `gpt-4o-mini` vs a bigger model - often the *smallest* gain per dollar |

That last row surprises people. In most RAG systems the model is not the bottleneck; retrieval is. Prove it with the table instead of guessing.

In [ ]:
# One change only: the search function.
hybrid_result = evaluate_rag(hybrid_search, label="hybrid, k=3", k=3)

print("\n\nCOMPARISON")
print(f"{'metric':<22}{'naive':<10}{'hybrid':<10}")
for key in ["retrieval_hit_rate", "faithfulness", "correctness", "refusal_accuracy", "seconds"]:
    a = baseline.get(key)
    b = hybrid_result.get(key)
    print(f"{key:<22}{str(a):<10}{str(b):<10}")

print("\nKeep the version with the better numbers. That is the entire method.")

## Step 16 - Reading the scorecard

Each metric points at a **different** fix. That is why we measure them separately.

| What you see | What it means | What to change |
|---|---|---|
| retrieval low, everything else fine | search is the bottleneck | chunking, hybrid, reranking, higher `k` |
| retrieval high, faithfulness low | the model is inventing on top of good chunks | strengthen the prompt, `temperature=0`, send fewer but better chunks |
| retrieval high, faithfulness high, correctness low | it answered a *different* question, or the reference is wrong | check the reference answer first, then the prompt |
| refusal accuracy low | it invents when it should refuse | rule 3 in the prompt; consider refusing when top score < a threshold |
| everything high, users still unhappy | your test set does not look like real traffic | log real questions and add them to the test set |

**The last row is the most common failure of evaluation itself.** A test set written by the developer tests the developer's imagination. Log real questions from day one, and grow the test set from actual failures.

## Online evaluation - after you ship

Everything above is *offline* evaluation: a fixed test set, run before release. In production you also need *online* signals:

| Signal | How to collect it | What it tells you |
|---|---|---|
| Thumbs up / down | a button next to every answer | the cheapest quality signal that exists |
| "I don't know" rate | count refusals | a sudden spike means retrieval broke |
| Retrieved chunk ids, logged per question | one log line | makes "it was wrong yesterday" debuggable |
| Escalation / handoff rate | support ticket created after a bot answer | the real business metric |
| Cost and latency per question | token counts | whether you can afford to keep it running |
| Unanswered question clusters | group refusals by topic | tells you which document to write next |

**Every thumbs-down should become a test case.** That loop - real failure → test set → fix → prove it with the scorecard - is what separates a demo from a product.

## The tools people actually use

You just built all of this by hand, which is the right way to learn it. In a job you would use one of these instead:

| Tool | What it is |
|---|---|
| **RAGAS** | the standard open-source RAG metric library: faithfulness, answer relevancy, context precision, context recall |
| **LangSmith** | traces every step plus datasets and experiments; strongest if you use LangChain / LangGraph |
| **DeepEval** | pytest-style LLM tests, so evaluation runs in CI like a unit test |
| **TruLens / Phoenix (Arize)** | tracing and evaluation dashboards, good for local and open-source stacks |
| **Vertex AI / Bedrock evaluation** | managed evaluation inside GCP / AWS |
| **Promptfoo** | fast side-by-side comparison of prompts and models from a config file |

They all implement the same four ideas you just wrote: a dataset, a target function, a set of scorers, and a comparison view. Knowing what the numbers *mean* is the transferable part - the library is a detail.

## What "good" looks like

Rough targets for an internal document assistant. Your bar depends on the cost of being wrong - a support bot and a medical assistant do not get the same threshold.

| Metric | Needs work | Shippable internally | Good |
|---|---|---|---|
| Retrieval hit rate @ 3 | < 0.80 | 0.90 | > 0.95 |
| Faithfulness | < 0.90 | 0.95 | > 0.98 |
| Correctness | < 0.75 | 0.85 | > 0.92 |
| Refusal accuracy | < 0.90 | 0.95 | ~1.00 |

Faithfulness has the highest bar on purpose. A system that is sometimes unhelpful is annoying. A system that is confidently wrong destroys trust in one screenshot.

## Summary of Parts 2 and 3

**Types of RAG** - every one is a fix for a specific failure, added to the six boxes you already built:

| Failure you actually see | Add |
|---|---|
| identifiers, codes and names are missed | hybrid search |
| the right chunk exists but ranks 9th | reranking |
| follow-ups and vague questions retrieve junk | query rewriting / multi-query / HyDE |
| answers are cut in half | parent-document expansion |
| wrong tenant, wrong year, wrong document type | metadata filters / self-query |
| one search is not enough | agentic RAG (grade and retry) |
| "how is A connected to B" | graph RAG |
| the answer is in a diagram | multimodal RAG |
| "how many", "average", "top 5" | SQL RAG |

**Evaluation** - four numbers, measured separately, on a saved test set:

| Metric | Catches |
|---|---|
| retrieval hit rate / MRR | search problems (measure this first, it is free) |
| faithfulness | hallucination |
| correctness | wrong answers |
| refusal accuracy | the dangerous failure |

**The five things to actually remember**

1. **Do naive RAG properly before adding anything.** Read your own chunks.
2. **Add types in cost order:** hybrid → reranking → rewriting → the rest.
3. **Measure retrieval before you touch the prompt.** It is free and it is usually the problem.
4. **Change one thing, re-run the test set, keep it only if the number went up.**
5. **Every real failure becomes a test case.** That is the whole job.

## Homework

1. Run `evaluate_retrieval(search, test_set, k=1)` then `k=5`. Where does hit rate stop improving? That is your `n`.
2. Re-chunk with `max_size=250`, rebuild the collection, and re-run `evaluate_rag`. Better or worse? Write down *why* before you look.
3. Write a `reranked_search(question, n=3)` that retrieves 8 and reranks to `n`, then score it with `evaluate_rag`. Compare all three retrieval types in one table.
4. Delete rule 3 from `SYSTEM_PROMPT` and re-run the evaluation. Watch `refusal_accuracy` collapse. Put it back.
5. Add 5 questions of your own to `test_set`, including at least 2 that should be refused. Notice how much harder it is to write good test questions than good code.
6. Add one deliberately wrong `reference` answer. Does `judge_correctness` catch it? If not, tighten the judge's prompt - this is how you learn to distrust judges appropriately.